In [78]:
import pymupdf
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama
import pandas as pd 

pdf_path = r"C:\Users\Asutosh\OneDrive\Documents\HOI ML codes\RAG\Impact of phase lag on synchronization.pdf"
pdf = pymupdf.open(pdf_path)

print(f"Number of pages in the pdf: {len(pdf)}")


Number of pages in the pdf: 7


In [79]:
#page extraction 

pages = []

for page_num, page in enumerate(pdf):
    text = page.get_text()
    pages.append({
        "page": page_num + 1,
        "text": text
    })

print(f"Pages extracted: {len(pages)}")
print(f"{pages[0]['text'][:4000]}")
print(f"{pages[1]['text'][:4000]}")

Pages extracted: 7
PHYSICAL REVIEW E 108, 034208 (2023)
Impact of phase lag on synchronization in frustrated Kuramoto model
with higher-order interactions
Sangita Dutta,1,* Abhijit Mondal,1 Prosenjit Kundu
,2,† Pitambar Khanra
,3,‡ Pinaki Pal,1,§ and Chittaranjan Hens4
1Department of Mathematics, National Institute of Technology, Durgapur 713209, India
2Dhirubhai Ambani Institute of Information and Communication Technology, Gandhinagar, Gujarat 382007, India
3Department of Mathematics, State University of New York at Buffalo, Buffalo 14260, USA
4Center for Computational Natural Science and Bioinformatics, International Institute of Informational Technology,
Gachibowli, Hyderabad 500032, India
(Received 4 May 2023; accepted 25 August 2023; published 18 September 2023)
The study of ﬁrst order transition (explosive synchronization) in an ensemble (network) of coupled oscillators
has been the topic of paramount interest among the researchers for more than one decade. Several frameworks
hav

In [80]:
#embedding 

model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = model.tokenizer

#chunking 
chunk_size = 120
chunk_overlap = 12

#chunks stored in a list here 
chunks = []

chunk_id = 0
for page_data in pages:
    page_number = page_data["page"]
    text = page_data["text"]

    #converting page text into token IDs
    token_ids = tokenizer.encode(text,
                                 add_special_tokens = False, truncation=False, verbose=False)

    start = 0

    while start < len(token_ids):
        end = start + chunk_size 

        chunk_token_ids = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_token_ids, skip_special_tokens = True)

        chunks.append({
            "chunk_id": chunk_id,
            "page": page_number,
            "text" : chunk_text, 
            "token_count": len(chunk_token_ids)       
        })

        chunk_id += 1

        #stopping conditon 
        if end >= len(token_ids):
            break
        #moving forward 
        start += chunk_size - chunk_overlap

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [81]:
chunks[0]

{'chunk_id': 0,
 'page': 1,
 'text': 'physical review e 108, 034208 ( 2023 ) impact of phase lag on synchronization in frustrated kuramoto model with higher - order interactions sangita dutta, 1, * abhijit mondal, 1 prosenjit kundu, 2, † pitambar khanra, 3, ‡ pinaki pal, 1, § and chittaranjan hens4 1department of mathematics, national institute of technology, durgapur 713209, india 2dhirubhai ambani institute of information and communication technology, gandhinagar, gujarat',
 'token_count': 120}

In [82]:
print(f"Total number of chunks created: {len(chunks)}")

Total number of chunks created: 93


In [83]:
for chunk in chunks:
    print(f"Chunk ID: {chunk['chunk_id']}, Page: {chunk['page']}, Text: {chunk['text'][:100]}...")

Chunk ID: 0, Page: 1, Text: physical review e 108, 034208 ( 2023 ) impact of phase lag on synchronization in frustrated kuramoto...
Chunk ID: 1, Page: 1, Text: ##i institute of information and communication technology, gandhinagar, gujarat 382007, india 3depar...
Chunk ID: 2, Page: 1, Text: oscillators has been the topic of paramount interest among the researchers for more than one decade....
Chunk ID: 3, Page: 1, Text: , phase frustration can promote explosive synchronization in a network. a low - dimensional model of...
Chunk ID: 4, Page: 1, Text: - served in natural and artiﬁcial systems [ 4 – 14 ]. a large set of coupled oscillators undergoes a...
Chunk ID: 5, Page: 1, Text: interactions. the phase of the ith oscillator is given by = ωi + k n j = 1 sin ( θj −θi ), i = 1, 2,...
Chunk ID: 6, Page: 1, Text: kuramoto model ( km ) [ 10, 15 ] described above shows second order ( continuous ) or ﬁrst order ( e...
Chunk ID: 7, Page: 1, Text: increased from a small value. in general, it sho

In [84]:
chunk_lengths = []
for chunk in chunks:

    ids = tokenizer.encode(chunk["text"], 
                           add_special_toekns = False, 
                           truncation = False, 
                           verbose = False)
    chunk_lengths. append(len(ids))

print(f"chunk_lengths: {chunk_lengths}")
print(f"Max Chunk Length: {max(chunk_lengths)}")
print(f"Min chunk Length: {min(chunk_lengths)}")

chunk_lengths: [122, 124, 122, 122, 122, 121, 122, 122, 122, 122, 122, 22, 122, 125, 124, 124, 122, 124, 121, 122, 122, 122, 125, 122, 101, 122, 124, 122, 122, 120, 120, 122, 122, 122, 117, 118, 123, 61, 122, 123, 122, 121, 121, 122, 124, 124, 122, 122, 122, 122, 122, 122, 113, 122, 122, 122, 122, 122, 122, 121, 121, 122, 122, 122, 62, 122, 122, 122, 122, 122, 122, 122, 122, 123, 122, 122, 122, 122, 122, 122, 124, 122, 81, 122, 122, 123, 122, 122, 122, 122, 122, 123, 38]
Max Chunk Length: 125
Min chunk Length: 22


In [85]:
#chunk embedding 
chunk_texts = [chunk["text"] for chunk in chunks
               ]

print("Number of chunks:", len(chunk_texts))

Number of chunks: 93


In [86]:
#embedding 
embeddings = model.encode(chunk_texts, 
                convert_to_numpy=True,
                normalize_embeddings = True)

print(f"Embedding shape:", embeddings.shape)

Embedding shape: (93, 384)


In [87]:
#query verification
query = "Which equation determines the onset of continous transitions?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Query Embedding shape:", query_embedding.shape)

Query Embedding shape: (384,)


In [106]:
#function to accept a query 
#this accepts the query from the user and retrives answers 

def retrieve(query_embedding, k_top=3):

    #cosine similarity 
    scores = embeddings @ query_embedding

    #indices pf top k-scores 

    top_indices = np.argsort(scores)[::-1][:k_top]

    #results in a list 
    results = []
    for idx in top_indices:
        results.append(
            {"chunk_id": chunks[idx]["chunk_id"],
             "page": chunks[idx]["page"], 
             "text": chunks[idx]["text"], 
             "score": float(scores[idx])
             })   

    return results        

In [107]:
#verification 
results = retrieve(query_embedding, k_top=3)

for result in results:
    print("="*80)
    print("Chunk_id:", result["chunk_id"])
    print()
    print(result["text"])
    print("Page:", result["page"])
    print("Similarity score:", result["score"])
    print()

Chunk_id: 46

point marking the onset of continuous transition, which gives k1 = k2. using it in the equation ( 13 ) the condition for the onset of continuous transition is obtained as k1 = k2 = 2 cos β. ( 14 ) now in particular, for ﬁxed k2 = 8, the onset of continuous transition occurs for k1 = 8 and β = cos−1 ( 1 / 4 ) = 1. 3181. in the numerical simulation, the continuous transition start also occurs exactly at this analytically determined value. on the other hand, for ﬁxed k1, as
Page: 4
Similarity score: 0.6300626397132874

Chunk_id: 66

of the ldm reveals the complex dependence of the transition scenario on the governing parameters. it is observed that the discontinuous transition is always as - sociated with a sn bifurcation. in the ﬁrst case mentioned above, the sn responsible for discontinuous transition is associated with a subcritical pitchfork bifurcation, while in the second case, sn bifurcation independently occur in a region of the parameter space. further, all the anal

In [108]:
#context 

def build_context(results):
    context_parts = []
    for result in results:
        context_parts.append(
            f"[Page{result['page']}\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)
    return context

In [109]:
#context 
context = build_context(results)
print(context)

[Page4
point marking the onset of continuous transition, which gives k1 = k2. using it in the equation ( 13 ) the condition for the onset of continuous transition is obtained as k1 = k2 = 2 cos β. ( 14 ) now in particular, for ﬁxed k2 = 8, the onset of continuous transition occurs for k1 = 8 and β = cos−1 ( 1 / 4 ) = 1. 3181. in the numerical simulation, the continuous transition start also occurs exactly at this analytically determined value. on the other hand, for ﬁxed k1, as

[Page6
of the ldm reveals the complex dependence of the transition scenario on the governing parameters. it is observed that the discontinuous transition is always as - sociated with a sn bifurcation. in the ﬁrst case mentioned above, the sn responsible for discontinuous transition is associated with a subcritical pitchfork bifurcation, while in the second case, sn bifurcation independently occur in a region of the parameter space. further, all the analyti - cally derived transition points determined from the l

In [110]:
#building prompt 

def build_prompt(query, context):

    prompt = f""" You are a very helpful assistant. You are gievn the following context. if the context does not contain 
    enough inofrmation, say: "The provided context does not contain enough information".

    Use only the information provided in the context to answer the question.Please reframe from using external knowledge or making vague assumptions.

    Please refer to page number as well. 
    
    Context: {context}
    Question: {query}
    Answer: 
    """

    return prompt 

prompt = build_prompt(query, context)

In [111]:
#generator Ollama 

def generate_answer(prompt):

    resposne = ollama.chat(
        model = "qwen2.5:1.5b",
        messages = [
                    {
            "role": "user", 
            "content": prompt
        }
        ]
    )

    return resposne["message"]["content"]

#rag integrated with Ollama 
  
def rag(query, k_top=3):

    #Retrieve relevant chunks
    results = retrieve(
        query_embedding,
        k_top=k_top
    )

    #context
    context = build_context(results)

    #prompt
    prompt = build_prompt(
        query,
        context
    )

    #answer
    answer = generate_answer(prompt)

    return answer, results

In [112]:
query = "What is the condition for the saddle-node bifurcation?"

answer, results = rag(query)

print("Answer:")
print(answer)

Answer:
The condition for the onset of continuous transition is given as \( k1 = k2 = 2 \cos \beta \).


In [113]:
evaluation_data = [

    {
        "id": "Q01",
        "category": "factual",
        "question": "What network size was used in the numerical simulations?",
        "gold_answer": "N = 10^3",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q02",
        "category": "factual",
        "question": "What numerical integration method was used?",
        "gold_answer": "Fourth-order Runge-Kutta method",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q03",
        "category": "parameter",
        "question": "What time step was used in the simulations?",
        "gold_answer": "delta t = 0.01",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q04",
        "category": "parameter",
        "question": "What distribution was used for the natural frequencies?",
        "gold_answer": "A Lorentzian distribution",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q05",
        "category": "parameter",
        "question": "What are K1 and K2 in the model?",
        "gold_answer": "K1 is the pairwise coupling strength and K2 is the higher-order interaction coupling strength.",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q06",
        "category": "numerical",
        "question": "What range of K1 was used for forward continuation?",
        "gold_answer": "K1 was increased from -1 to 12.",
        "gold_pages": [2],
        "answerable": True
    },

    {
        "id": "Q07",
        "category": "conceptual",
        "question": "For fixed K2 = 8, what happens to the synchronization transition as phase lag beta increases?",
        "gold_answer": "The discontinuous transition becomes continuous as beta increases, with continuous transition above approximately beta = 1.318.",
        "gold_pages": [2, 4],
        "answerable": True
    },

    {
        "id": "Q08",
        "category": "conceptual",
        "question": "What happens when K2 is varied at fixed K1 = 2.5 as beta increases?",
        "gold_answer": "For small beta the transition is continuous, while for beta greater than about 0.6435 discontinuous transitions appear.",
        "gold_pages": [3, 4, 5],
        "answerable": True
    },

    {
        "id": "Q09",
        "category": "equation",
        "question": "What condition determines the saddle-node bifurcation?",
        "gold_answer": "(K1 + K2)^2 cos(beta) = 8 K2",
        "gold_pages": [4],
        "answerable": True
    },

    {
        "id": "Q10",
        "category": "equation",
        "question": "What condition determines the onset of continuous transition?",
        "gold_answer": "K1 = K2 = 2/cos(beta)",
        "gold_pages": [4],
        "answerable": True
    },

    {
        "id": "Q11",
        "category": "analytical",
        "question": "What critical phase lag is obtained for K2 = 8 at the onset of continuous transition?",
        "gold_answer": "beta = arccos(1/4) approximately 1.3181",
        "gold_pages": [4],
        "answerable": True
    },

    {
        "id": "Q12",
        "category": "analytical",
        "question": "What role does the saddle-node bifurcation play in the discontinuous transition?",
        "gold_answer": "The discontinuous transition is associated with the existence of a saddle-node bifurcation.",
        "gold_pages": [4, 6],
        "answerable": True
    },

    {
        "id": "Q13",
        "category": "conceptual",
        "question": "What is the main surprising effect of higher-order interactions on phase frustration?",
        "gold_answer": "In the presence of higher-order interactions, sufficiently large phase lag can promote discontinuous synchronization rather than suppress it.",
        "gold_pages": [3, 5, 6],
        "answerable": True
    },

    {
        "id": "Q14",
        "category": "unanswerable",
        "question": "What machine learning optimizer was used to train the model?",
        "gold_answer": None,
        "gold_pages": [],
        "answerable": False
    },

    {
        "id": "Q15",
        "category": "unanswerable",
        "question": "What GPU model was used for the numerical simulations?",
        "gold_answer": None,
        "gold_pages": [],
        "answerable": False
    }
]

In [114]:
for item in evaluation_data:
    print(
        item["id"],
        "gold_pages =", item["gold_pages"],
        "type =", type(item["gold_pages"]),
        "answerable =", item["answerable"]
    )

Q01 gold_pages = [2] type = <class 'list'> answerable = True
Q02 gold_pages = [2] type = <class 'list'> answerable = True
Q03 gold_pages = [2] type = <class 'list'> answerable = True
Q04 gold_pages = [2] type = <class 'list'> answerable = True
Q05 gold_pages = [2] type = <class 'list'> answerable = True
Q06 gold_pages = [2] type = <class 'list'> answerable = True
Q07 gold_pages = [2, 4] type = <class 'list'> answerable = True
Q08 gold_pages = [3, 4, 5] type = <class 'list'> answerable = True
Q09 gold_pages = [4] type = <class 'list'> answerable = True
Q10 gold_pages = [4] type = <class 'list'> answerable = True
Q11 gold_pages = [4] type = <class 'list'> answerable = True
Q12 gold_pages = [4, 6] type = <class 'list'> answerable = True
Q13 gold_pages = [3, 5, 6] type = <class 'list'> answerable = True
Q14 gold_pages = [] type = <class 'list'> answerable = False
Q15 gold_pages = [] type = <class 'list'> answerable = False


In [115]:
test_question = evaluation_data[0]["question"]

test_embedding = model.encode(
    test_question,
    convert_to_numpy=True,
    normalize_embeddings=True
)

test_results = retrieve(
    test_embedding,
    k_top=5
)

print(type(test_results))
print(test_results)

<class 'list'>
[{'chunk_id': 18, 'page': 2, 'text': '67 ]. ii. model description and numerical simuations the dynamics of a phase frustrated undirected simplicial complex of n number of nodes with global connectivity is governed by the following system of equations : = ωi + k1 n n j = 1 sin ( θj −θi −β ) + k2 n2 n j = 1 n l = 1 sin ( 2θj −θl −θi −β ), i = 1, 2,..., n. ( 3 ) this is the generalization of the classical sakaguchi -', 'score': 0.43347904086112976}, {'chunk_id': 3, 'page': 1, 'text': ', phase frustration can promote explosive synchronization in a network. a low - dimensional model of the network in the thermodynamic limit is derived using the ott - antonsen ansatz to explain this surprising result. analytical treatment of the low - dimensional model, including bifurcation analysis, explains the apparent counter intuitive result quite clearly. doi : 10. 1103 / physreve. 108. 034208 i. introduction synchronization [ 1 – 3 ] is a captivating phenomenon ob - served in natural a

In [116]:
def evaluate_retrieval(evaluation_data, k_max=5):

    retrieval_records = []

    for item in evaluation_data:

        # Skip unanswerable questions for retrieval evaluation
        if not item["answerable"]:
            continue

        question = item["question"]
        gold_pages = item["gold_pages"]

        #query embeding 
        query_embedding = model.encode(
            question,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        # Retrieve top-k chunks
        results = retrieve(
            query_embedding,
            k_top=k_max
        )

        # Pages returned by the retriever
        retrieved_pages = [
            result["page"]
            for result in results
        ]

        # Find rank of first relevant result
        first_relevant_rank = None

        for rank, result in enumerate(results, start=1):

            if result["page"] in gold_pages:
                first_relevant_rank = rank
                break

        #Hit@1
        hit_at_1 = int(
            any(
                page in gold_pages
                for page in retrieved_pages[:1]
            )
        )

        #Hit@3
        hit_at_3 = int(
            any(
                page in gold_pages
                for page in retrieved_pages[:3]
            )
        )

        #hit at 5
        hit_at_5 = int(
            any(
                page in gold_pages
                for page in retrieved_pages[:5]
            )
        )

        #reciprocal rank
        if first_relevant_rank is None:
            reciprocal_rank = 0.0
        else:
            reciprocal_rank = 1 / first_relevant_rank

        # Store results
        retrieval_records.append({
            "id": item["id"],
            "category": item["category"],
            "question": question,
            "gold_pages": gold_pages,
            "retrieved_pages": retrieved_pages,
            "first_relevant_rank": first_relevant_rank,
            "hit_at_1": hit_at_1,
            "hit_at_3": hit_at_3,
            "hit_at_5": hit_at_5,
            "reciprocal_rank": reciprocal_rank
        })

    return retrieval_records

In [117]:
retrieval_records = evaluate_retrieval(
    evaluation_data,
    k_max=5
)

print(type(retrieval_records))
print(len(retrieval_records))

<class 'list'>
13


In [118]:
retrieval_records[0]

{'id': 'Q01',
 'category': 'factual',
 'question': 'What network size was used in the numerical simulations?',
 'gold_pages': [2],
 'retrieved_pages': [2, 1, 3, 6, 7],
 'first_relevant_rank': 1,
 'hit_at_1': 1,
 'hit_at_3': 1,
 'hit_at_5': 1,
 'reciprocal_rank': 1.0}

In [119]:
#converting the records into dataframe  
retrieval_df = pd.DataFrame(retrieval_records)
print(retrieval_df)

     id    category                                           question  \
0   Q01     factual  What network size was used in the numerical si...   
1   Q02     factual        What numerical integration method was used?   
2   Q03   parameter        What time step was used in the simulations?   
3   Q04   parameter  What distribution was used for the natural fre...   
4   Q05   parameter                   What are K1 and K2 in the model?   
5   Q06   numerical  What range of K1 was used for forward continua...   
6   Q07  conceptual  For fixed K2 = 8, what happens to the synchron...   
7   Q08  conceptual  What happens when K2 is varied at fixed K1 = 2...   
8   Q09    equation  What condition determines the saddle-node bifu...   
9   Q10    equation  What condition determines the onset of continu...   
10  Q11  analytical  What critical phase lag is obtained for K2 = 8...   
11  Q12  analytical  What role does the saddle-node bifurcation pla...   
12  Q13  conceptual  What is the main 

In [121]:
#retireval metrices 
hit_at_1 = retrieval_df["hit_at_1"].mean()
hit_at_3 = retrieval_df["hit_at_3"].mean()
hit_at_5 = retrieval_df["hit_at_5"].mean()
mrr = retrieval_df["reciprocal_rank"].mean()

print(f"Hit@1 : {hit_at_1:.3f}")
print(f"Hit@3 : {hit_at_3:.3f}")
print(f"Hit@5 : {hit_at_5:.3f}")
print(f"MRR   : {mrr:.3f}")

Hit@1 : 0.538
Hit@3 : 0.692
Hit@5 : 0.846
MRR   : 0.624


In [123]:
#inspecting failures 
failed_retrievals = retrieval_df[
    retrieval_df["hit_at_3"] == 0
]

failed_retrievals[
    [
        "id",
        "category",
        "question",
        "gold_pages",
        "retrieved_pages"
    ]
]

,id,category,question,gold_pages,retrieved_pages
1,Q02,factual,What numerical integration method was used?,[2],"[4, 3, 6, 4, 3]"
3,Q04,parameter,What distribution was used for the natural fre...,[2],"[1, 1, 3, 1, 2]"
6,Q07,conceptual,"For fixed K2 = 8, what happens to the synchron...","[2, 4]","[5, 3, 5, 1, 3]"
12,Q13,conceptual,What is the main surprising effect of higher-o...,"[3, 5, 6]","[2, 1, 1, 3, 1]"


In [125]:
lower_rank_success = retrieval_df[
    (retrieval_df["hit_at_1"] == 0) &
    (retrieval_df["hit_at_3"] == 1)
]

lower_rank_success[
    [
        "id",
        "category",
        "question",
        "gold_pages",
        "retrieved_pages",
        "first_relevant_rank"
    ]
]

,id,category,question,gold_pages,retrieved_pages,first_relevant_rank
2,Q03,parameter,What time step was used in the simulations?,[2],"[6, 4, 2, 2, 2]",3.0
4,Q05,parameter,What are K1 and K2 in the model?,[2],"[5, 1, 2, 5, 3]",3.0
